### Train MNIST ResNet
- If running on Colab compute, clones repo to gain access to all files
- Login to WandB
- Downloads MNIST dataset
- Runs training loop, logs and saves weights to WandB

In [ ]:
# Colab Setup: clone repo and install requirements
import os, sys

# check if running in Colab
IN_COLAB = "google.colab" in sys.modules
print(f"Running on Colab:{IN_COLAB}")

if IN_COLAB:
    if not os.path.exists("vis10n"):
        print("Cloning full repo")
        !git clone https://github.com/simchaorgel/vis10n.git
    !cd vis10n && git pull
    sys.path.append("vis10n")
    # Download requirements
    print("Downloading requirements")
    !pip install -q -r vis10n/requirements.txt

In [ ]:
# Main imports
from torchvision import models, datasets, transforms
import torch as t
from dataclasses import dataclass
from torch.utils.data import Subset, DataLoader
import torch.nn.functional as F
from tqdm.notebook import tqdm
import wandb

In [ ]:
# Local imports: this cell imports the necessary files
# either locally or from colabs repo clone
from models.resnet_mnist_1 import Model
import optimizers

In [ ]:
# Check if we have access to a GPU
device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
print(device)

In [ ]:
# Wandb login
wandb.login()

In [ ]:
# Load datasets (train and test sets)
# Transform
MNIST_TRANSFORM = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(0.1307, 0.3081),
    ]
)

def get_mnist(trainset_size: int = 10_000, testset_size: int = 1_000) -> tuple[Subset, Subset]:
    """Returns a subset of MNIST training data."""
    # Get or download original datasets, which are downloaded to "data/" either locally or on colab
    mnist_trainset = datasets.MNIST("data/", train=True, download=True, transform=MNIST_TRANSFORM)
    mnist_testset = datasets.MNIST("data/", train=False, download=True, transform=MNIST_TRANSFORM)

    # Return a subset of the original datasets
    mnist_trainset = Subset(mnist_trainset, indices=range(trainset_size))
    mnist_testset = Subset(mnist_testset, indices=range(testset_size))
    
    return mnist_trainset, mnist_testset

In [ ]:
# Training args
#####################################################
@dataclass
class ResNetTrainingArgs:
    trainset_size: int = 60000
    testset_size: int = 2000
    batch_size: int = 64
    epochs: int = 4
    optimizer: str = "AdamW"
    learning_rate: float = 1e-3
    n_classes: int = 10
    save_model: bool = True

In [ ]:
# Train
#####################################################
def train(args: ResNetTrainingArgs) -> Model:
    """
    Trains ResNet on MNIST, logging to wandb. Returns the trained model, also saves to wandb
    """
    try:
        # Wandb init
        wandb.init(entity="simchao", project="mnist-resnet", config=args)
        # Create ResNet instance
        model = Model().to(device)
        #wandb.watch(model, log="all", log_freq=50)
        # Get data
        trainset, testset = get_mnist(args.trainset_size, args.testset_size)
        train_data = DataLoader(trainset, batch_size=args.batch_size, shuffle=True)
        test_data = DataLoader(testset, batch_size=args.batch_size, shuffle=False)
        # Define optimizer
        optimizer = optimizers.get_optimizer(args.optimizer)(model.parameters(), lr=args.learning_rate)
        # Logging lists
        examples_seen = 0

        # For every epoch:
        for epoch in range(args.epochs):
            epoch_accurate_predictions = 0 # For validating
            pbar = tqdm(train_data)

            # Train - loop for every batch in this epoch
            model.train()
            for imgs, labels in pbar:
                # Move data to device, perform forward pass
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                # Compute loss, perform backward pass
                loss = F.cross_entropy(logits, labels)
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

                # Update logs & progress bar
                examples_seen += imgs.shape[0]
                wandb.log({"loss": loss.item()}, step=examples_seen)
                pbar.set_postfix(epoch=f"{epoch + 1}/{args.epochs}", loss=f"{loss:.3f}")

            # Test to get accuracy
            model.eval()
            for imgs, labels in test_data:
                imgs, labels = imgs.to(device), labels.to(device)
                with t.inference_mode():
                    logits = model(imgs)

                # Calculate accuracy
                # Get max logit, which is treated as the models guess
                predictions = t.argmax(logits, dim=1)
                # Create tensor with 1 if the prediction was correct, 0 if not, for every img
                accuracy_bools = predictions == labels
                # Number of accurate guesses
                epoch_accurate_predictions += accuracy_bools.sum().item()

            # Before completing the epoch
            # Add accuracy percentage to list
            epoch_accuracy = epoch_accurate_predictions / len(testset)
            wandb.log({"accuracy": epoch_accuracy}, step=examples_seen)

        if args.save_model:
            t.save(model.state_dict(), "mnist_resnet.pt")
            artifact = wandb.Artifact("mnist_resnet", type="model")
            artifact.add_file("mnist_resnet.pt")
            wandb.log_artifact(artifact)
            
        wandb.finish()
        return model
    
    except Exception:
        wandb.finish()
        raise


args = ResNetTrainingArgs()
model = train(args)